# 支援向量機(SVM)實驗:最大間隔、軟間隔與核函數

**這份筆記在做什麼?**

延續前兩份實驗(線性迴歸、邏輯迴歸),這次登場的是經典分類器 **SVM(Support Vector Machine)**。
整份實驗圍繞三個問題:

1. **與傳統演算法對比,SVM 究竟能帶來什麼樣的效果?** —— 一樣能分對的線有無限多條,SVM 挑「最大間隔」的那條
2. **這麼強的演算法一定會導致過擬合,如何解決?** —— 軟間隔,用超參數 C 控制
3. **如果只是做線性分類,好像輪不到 SVM 登場 —— 核函數才是它的強大之處!** —— 多項式核與高斯核

**學習地圖:**

| 部分 | 主題 | 回答的問題 |
|:---:|---|---|
| 1 | 最大間隔 | 都能分對的線那麼多,哪條最好?支持向量是什麼? |
| 2 | 標準化的影響 | 為什麼 SVM 訓練前要先標準化? |
| 3 | 軟間隔與超參數 C | 資料裡有異常點怎麼辦?C 調大調小差在哪? |
| 4 | 非線性 SVM | 資料彎的怎麼辦?(特徵升維) |
| 5 | 核函數 | 不真的升維,也能有升維的效果?(多項式核、高斯核、$\gamma$) |

> 💡 請**由上往下依序執行**每個 cell(Shift + Enter):前面定義的變數與函式,後面會繼續使用。

## 0. 環境準備

In [1]:
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

# 統一圖表字體大小
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12

import warnings
warnings.filterwarnings('ignore')   # 忽略警告訊息,保持輸出乾淨

## 1. 支援向量機帶來的效果:最大間隔

### 1.1 問題:能分對的線有無限多條,哪一條最好?

用鳶尾花資料集做一個**線性可分**的二分類:只取 Setosa 和 Versicolor 兩類(這兩類分得很開),
特徵一樣用花瓣長度與寬度。

先訓練一個**線性 SVM**:`kernel='linear'`;`C=inf` 表示**硬間隔**(hard margin)——
完全不允許任何樣本越界,適合這種兩類完全分得開的情況(C 的意義第 3 部分細講)。

In [2]:
from sklearn.svm import SVC
from sklearn import datasets

iris = datasets.load_iris()
X = iris['data'][:, (2, 3)]              # 兩個特徵:花瓣長度、花瓣寬度
y = iris['target']

setosa_or_versicolor = (y == 0) | (y == 1)   # 只留類別 0 和 1(這兩類線性可分)
X = X[setosa_or_versicolor]
y = y[setosa_or_versicolor]

svm_clf = SVC(kernel='linear', C=float('inf'))   # 線性核 + 硬間隔
svm_clf.fit(X, y)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",inf
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


### 1.2 畫決策邊界的工具函式

訓練好的線性 SVM 學到權重 $\mathbf{w}$ 和截距 $b$,平面上的分界就是:

- **決策邊界(黑實線)**:$\mathbf{w}^T \mathbf{x} + b = 0$ 的位置
- **間隔邊線(黑虛線)**:$\mathbf{w}^T \mathbf{x} + b = \pm 1$ 的位置,即決策邊界往兩側平移
- **支持向量(紅圈)**:恰好壓在間隔邊線上的樣本,可從 `svm_clf.support_vectors_` 取得

In [3]:
def plot_svc_decision_boundary(svm_clf, xmin, xmax, sv=True):
    '''畫出線性 SVM 的決策邊界、間隔邊線,並圈出支持向量'''
    w = svm_clf.coef_[0]        # 權重 [w0, w1]
    b = svm_clf.intercept_[0]   # 截距
    x0 = np.linspace(xmin, xmax, 200)
    decision_boundary = -w[0]/w[1] * x0 - b/w[1]   # 由 w0*x0 + w1*x1 + b = 0 解出 x1
    margin = 1/w[1]                                # 間隔邊線 = 邊界沿 x1 方向平移 ±1/w1(對應 w^T x + b = ±1)
    gutter_up = decision_boundary + margin
    gutter_down = decision_boundary - margin
    if sv:
        svs = svm_clf.support_vectors_             # 支持向量:壓在間隔邊線上的樣本
        plt.scatter(svs[:, 0], svs[:, 1], s=180, facecolors='#FFAAAA')
    plt.plot(x0, decision_boundary, 'k-', linewidth=2)   # 黑實線:決策邊界
    plt.plot(x0, gutter_up, 'k--', linewidth=2)          # 黑虛線:間隔邊線
    plt.plot(x0, gutter_down, 'k--', linewidth=2)

### 1.3 對比實驗:隨手畫的分界線 vs SVM

左圖放三條「一般模型」可能給出的分界線(這裡直接手寫三條直線方程式模擬),右圖放 SVM 學到的邊界:

In [ ]:
# 手寫三條分界線,模擬「一般模型」可能的結果
x0 = np.linspace(0, 5.5, 200)
pred_1 = 5*x0 - 20        # 綠虛線:根本分錯邊
pred_2 = x0 - 1.8         # 紫線:分對了,但貼著樣本
pred_3 = 0.1 * x0 + 0.5   # 紅線:分對了,但貼著另一側樣本

plt.figure(figsize=(14, 4))

plt.subplot(121)
plt.plot(X[:, 0][y == 1], X[:, 1][y == 1], 'bs')   # 藍方塊:Versicolor
plt.plot(X[:, 0][y == 0], X[:, 1][y == 0], 'ys')   # 黃方塊:Setosa
plt.plot(x0, pred_1, 'g--', linewidth=2)
plt.plot(x0, pred_2, 'm-', linewidth=2)
plt.plot(x0, pred_3, 'r-', linewidth=2)
plt.axis([0, 5.5, 0, 2])

plt.subplot(122)
plot_svc_decision_boundary(svm_clf, 0, 5.5)        # SVM 學到的邊界
plt.plot(X[:, 0][y == 1], X[:, 1][y == 1], 'bs')
plt.plot(X[:, 0][y == 0], X[:, 1][y == 0], 'ys')
plt.axis([0, 5.5, 0, 2])
plt.show()

**觀察結果:**

- 左圖:綠虛線直接分錯;紫線和紅線雖然全分對,但**貼著某一側的樣本** ——
  新來的資料只要稍微偏一點就會被判錯,泛化能力差
- 右圖:SVM 的黑實線**離兩類都最遠**,兩條虛線之間的距離就是**間隔(margin)**;
  SVM 的目標正是「在分對的前提下,讓間隔最大」—— 所以又稱**最大間隔分類器**
- 紅圈的**支持向量**是唯一「撐住」邊界的樣本:其他點就算全部拿掉,邊界也不會變 ——
  這就是「支援向量機」名字的由來

> 💡 間隔越大,對新資料的容錯空間越大,泛化能力越好。

## 2. 資料標準化的影響

SVM 追求「距離」上的最大間隔 —— 而距離對**特徵尺度**非常敏感。
下圖左(Unscaled):$x_1$ 的數值範圍遠大於 $x_0$,間隔幾乎完全由 $x_1$ 主導,邊界又扁又窄;
下圖右(Scaled):標準化後兩個特徵尺度一致,間隔明顯變大、邊界也合理得多:

![標準化前後的決策邊界與間隔](./img/2.png)

> 💡 和前兩份實驗的結論一致:**拿到資料先標準化**,對 SVM 尤其重要(之後的程式都會把 `StandardScaler` 放進 Pipeline)。

## 3. 軟間隔(Soft Margin)

### 3.1 如果不加入軟間隔,會遇到哪些問題?

硬間隔要求「每一個樣本都乖乖待在自己那側、且不進入間隔」。只要資料裡混進**異常點(outlier)**就出事:

![硬間隔遇到異常點的兩種困境](./img/3.png)

- 左圖:黃色異常點混進了藍色陣營 —— 兩類根本無法線性分開,硬間隔**無解(Impossible!)**
- 右圖:異常點雖然還能分,卻把間隔**擠得非常窄** —— 為了遷就一個怪點,犧牲了整體的泛化能力

**軟間隔**的想法:允許少數樣本越線(進入間隔、甚至跑錯邊),換取更寬、更穩的間隔。
可以使用**超參數 C 控制軟間隔程度**:C 越大,對越線的懲罰越重(越接近硬間隔);C 越小,越寬容。

> 📝 眼熟嗎?這正是邏輯迴歸篇看過的同一個 `C`(正規化強度的倒數)。

先用 sklearn 的 `LinearSVC` 訓練一個軟間隔分類器(做「是不是 Virginica」二分類,三類都在、且兩類有重疊,正好需要軟間隔):

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

X = iris['data'][:, (2, 3)]                       # 花瓣長度、花瓣寬度
y = (iris['target'] == 2).astype(np.float64)      # 標籤變換:是 Virginica → 1,其他 → 0

svm_clf = Pipeline([
    ('std', StandardScaler()),     # 第 2 部分的教訓:先標準化
    ('linear_svc', LinearSVC(C=1))
])
svm_clf.fit(X, y)

In [ ]:
svm_clf.predict([[5.5, 1.7]])   # 測一筆:花瓣長 5.5、寬 1.7 → 預測為 1(是 Virginica)

### 3.2 比較不同 C 值所帶來的效果差異

訓練兩個模型:`C=1`(寬鬆)和 `C=100`(嚴格),把兩者的決策邊界畫在原始座標上對比。

In [ ]:
scaler = StandardScaler()
svm_clf1 = LinearSVC(C=1, random_state=42)     # 寬鬆:容忍較多越線
svm_clf2 = LinearSVC(C=100, random_state=42)   # 嚴格:重罰越線

scaled_svm_clf1 = Pipeline([
    ('std', scaler),
    ('linear_svc', svm_clf1)
])
scaled_svm_clf2 = Pipeline([
    ('std', scaler),
    ('linear_svc', svm_clf2)
])
scaled_svm_clf1.fit(X, y)
scaled_svm_clf2.fit(X, y)

一個小麻煩:模型是在**標準化後的空間**學到 $\mathbf{w}, b$ 的,想畫回**原始座標**,
得把參數「反標準化」換算回去(下面這段就是在做座標換算,看不懂細節沒關係,知道目的即可):

In [ ]:
# 把「標準化空間」的參數換算回「原始特徵空間」,才能疊在原始資料上畫圖
b1 = svm_clf1.decision_function([-scaler.mean_ / scaler.scale_])   # 原始原點在標準化空間的位置 → 換算截距
b2 = svm_clf2.decision_function([-scaler.mean_ / scaler.scale_])
w1 = svm_clf1.coef_[0] / scaler.scale_                             # 權重除以每個特徵的標準差
w2 = svm_clf2.coef_[0] / scaler.scale_
svm_clf1.intercept_ = np.array([b1])
svm_clf2.intercept_ = np.array([b2])
svm_clf1.coef_ = np.array([w1])
svm_clf2.coef_ = np.array([w2])

In [ ]:
plt.figure(figsize=(14, 4.2))

plt.subplot(121)
plt.plot(X[:, 0][y == 1], X[:, 1][y == 1], "g^", label="Iris-Virginica")
plt.plot(X[:, 0][y == 0], X[:, 1][y == 0], "bs", label="Iris-Versicolor")
plot_svc_decision_boundary(svm_clf1, 4, 6, sv=False)   # LinearSVC 不提供支持向量,只畫邊界
plt.xlabel("Petal length", fontsize=14)
plt.ylabel("Petal width", fontsize=14)
plt.legend(loc="upper left", fontsize=14)
plt.title("$C = {}$".format(svm_clf1.C), fontsize=16)
plt.axis([4, 6, 0.8, 2.8])

plt.subplot(122)
plt.plot(X[:, 0][y == 1], X[:, 1][y == 1], "g^")
plt.plot(X[:, 0][y == 0], X[:, 1][y == 0], "bs")
plot_svc_decision_boundary(svm_clf2, 4, 6, sv=False)
plt.xlabel("Petal length", fontsize=14)
plt.title("$C = {}$".format(svm_clf2.C), fontsize=16)
plt.axis([4, 6, 0.8, 2.8])
plt.show()

**觀察結果:**

- **右側(C=100,較高)**:分類器會盡量減少誤分類,但最終得到**較小的間隔** —— 邊界被少數難分的點牽著走
- **左側(C=1,較低)**:**間隔大得多**,但許多實例會出現在間隔之內 —— 模型更寬容、更平穩

> 💡 C 是 SVM 最重要的超參數之一:**模型過擬合時,試著調小 C**(放寬限制);欠擬合時調大 C。

## 4. 非線性支援向量機

### 4.1 升維:讓「不可分」變「可分」

線性 SVM 再強,遇到線性不可分的資料就沒轍了。出路和多項式迴歸一樣:**對特徵做非線性變換(升維)**。

看一個一維的例子:9 個點排成一直線,兩端是藍色、中間是綠色 —— 在一維上,**一個切點**不可能分開它們(左圖)。
但幫每個點加上第二個特徵 $x_2 = x_1^2$ 之後,點被抬到拋物線上 —— **一條直線**(紅色虛線)就輕鬆切開了(右圖):

In [ ]:
X1D = np.linspace(-4, 4, 9).reshape(-1, 1)   # 一維資料:-4 ~ 4 的 9 個點
X2D = np.c_[X1D, X1D**2]                     # 升維:加上平方項當第二個特徵
y = np.array([0, 0, 1, 1, 1, 1, 1, 0, 0])    # 兩端是類別 0,中間是類別 1

plt.figure(figsize=(11, 4))

plt.subplot(121)                             # 左圖:原始一維資料
plt.grid(True, which='both')
plt.axhline(y=0, color='k')
plt.plot(X1D[:, 0][y == 0], np.zeros(4), "bs")
plt.plot(X1D[:, 0][y == 1], np.zeros(5), "g^")
plt.gca().get_yaxis().set_ticks([])
plt.xlabel(r"$x_1$", fontsize=20)
plt.axis([-4.5, 4.5, -0.2, 0.2])

plt.subplot(122)                             # 右圖:升維到 (x1, x1^2)
plt.grid(True, which='both')
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.plot(X2D[:, 0][y == 0], X2D[:, 1][y == 0], "bs")
plt.plot(X2D[:, 0][y == 1], X2D[:, 1][y == 1], "g^")
plt.xlabel(r"$x_1$", fontsize=20)
plt.ylabel(r"$x_2$", fontsize=20, rotation=0)
plt.gca().get_yaxis().set_ticks([0, 4, 8, 12, 16])
plt.plot([-4.5, 4.5], [6.5, 6.5], "r--", linewidth=3)   # 升維後,一條直線就能分開
plt.axis([-4.5, 4.5, -1, 17])

plt.subplots_adjust(right=1)
plt.show()

### 4.2 創建一份有點難度的資料

`make_moons`:兩個彎月互相咬合的經典非線性資料集,直線肯定分不開。

In [ ]:
from sklearn.datasets import make_moons
X, y = make_moons(n_samples=100, noise=0.15, random_state=42)   # 100 筆、加一點雜訊

def plot_dataset(X, y, axes):
    '''畫出兩類資料點'''
    plt.plot(X[:, 0][y == 0], X[:, 1][y == 0], "bs")
    plt.plot(X[:, 0][y == 1], X[:, 1][y == 1], "g^")
    plt.axis(axes)
    plt.grid(True, which='both')
    plt.xlabel(r"$x_1$", fontsize=20)
    plt.ylabel(r"$x_2$", fontsize=20, rotation=0)

plot_dataset(X, y, [-1.5, 2.5, -1, 1.5])
plt.show()

### 4.3 多項式特徵 + 線性 SVM

套路和多項式迴歸完全一樣:`PolynomialFeatures` 升維 → 標準化 → 線性分類器,用 Pipeline 串起來。
(`loss="hinge"` 是 SVM 的標準損失函數:分對且在間隔外 → 損失 0;越靠近邊界、越線 → 損失越大)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

polynomial_svm_clf = Pipeline([
    ("poly_features", PolynomialFeatures(degree=3)),   # 升維:做到三次方特徵
    ("scaler", StandardScaler()),
    ("svm_clf", LinearSVC(C=10, loss="hinge"))
])

polynomial_svm_clf.fit(X, y)

畫出它的決策範圍。畫法沿用邏輯迴歸篇的套路:**meshgrid 鋪網格 → 每點 predict → contourf 填色**:

In [ ]:
def plot_predictions(clf, axes):
    '''把分類器在整個平面上的預測結果用底色畫出來'''
    x0s = np.linspace(axes[0], axes[1], 100)
    x1s = np.linspace(axes[2], axes[3], 100)
    x0, x1 = np.meshgrid(x0s, x1s)            # 鋪 100×100 的網格
    X = np.c_[x0.ravel(), x1.ravel()]         # 整合成測試點清單
    y_pred = clf.predict(X).reshape(x0.shape)  # 每一點的預測類別
    plt.contourf(x0, x1, y_pred, cmap=plt.cm.brg, alpha=0.2)   # 依類別填上底色

plot_predictions(polynomial_svm_clf, [-1.5, 2.5, -1, 1.5])
plot_dataset(X, y, [-1.5, 2.5, -1, 1.5])
plt.show()

**觀察結果:** 三次多項式特徵讓「線性」SVM 畫出了一條漂亮的彎曲邊界,把兩個彎月分開了。

## 5. SVM 中的核函數技巧

### 5.1 多項式核(Polynomial Kernel)

4.3 的做法有個隱憂:degree 一高,`PolynomialFeatures` 產生的特徵數會**爆炸式成長**,算不動。

SVM 的殺手鐧 —— **核技巧(kernel trick)**:數學上可以證明,SVM 訓練過程只需要樣本之間的**內積**;
核函數能「直接算出升維後的內積」,**而不必真的生成高維特徵**。
效果等同於升維,計算量卻小得多。用 `SVC(kernel="poly")` 直接指定多項式核:

In [ ]:
poly_kernel_svm_clf = Pipeline([
        ("scaler", StandardScaler()),
        ("svm_clf", SVC(kernel="poly", degree=3, coef0=1, C=5))     # 三次多項式核
    ])

poly_kernel_svm_clf.fit(X, y)

In [ ]:
poly100_kernel_svm_clf = Pipeline([
        ("scaler", StandardScaler()),
        ("svm_clf", SVC(kernel="poly", degree=10, coef0=100, C=5))  # 十次多項式核,coef0 調大
    ])

poly100_kernel_svm_clf.fit(X, y)

In [ ]:
plt.figure(figsize=(11, 4))

plt.subplot(121)
plot_predictions(poly_kernel_svm_clf, [-1.5, 2.5, -1, 1.5])
plot_dataset(X, y, [-1.5, 2.5, -1, 1.5])
plt.title(r"$d=3, r=1, C=5$", fontsize=18)

plt.subplot(122)
plot_predictions(poly100_kernel_svm_clf, [-1.5, 2.5, -1, 1.5])
plot_dataset(X, y, [-1.5, 2.5, -1, 1.5])
plt.title(r"$d=10, r=100, C=5$", fontsize=18)

plt.show()

**觀察結果:** 左圖(degree=3)邊界平順;右圖(degree=10)邊界更彎、貼資料貼得更緊 —— degree 越高越容易過擬合。

> 📝 `coef0`(圖標題中的 $r$)表示**偏移項**:它控制模型受高次項影響的程度,調大 coef0 會放大高次項的作用。

### 5.2 高斯核函數(RBF):利用「相似度」來轉換特徵

多項式核之外,另一個更常用的核:**高斯核(RBF, Radial Basis Function)**。它換一種升維思路 ——
**新特徵 = 這個樣本跟某些「地標(landmark)」有多像**。

用一維小例子走一遍:

* 選取一份一維資料集,並在 $x_1 = -2$ 和 $x_1 = 1$ 處為其新增兩個高斯函數(地標)
* 接下來,將相似度函數定義為 $\gamma = 0.3$ 的徑向基底函數(RBF):

![RBF 相似度函數公式](./img/5.png)

公式讀法:$\mathbf{x}$ 是樣本、$\ell$ 是地標、$\|\mathbf{x}-\ell\|$ 是兩者距離 ——
**距離越近,相似度越接近 1;距離越遠,相似度越趨近 0**。

例如 **$x_1 = -1$**:它距離第一個地標 **1**,距離第二個地標 **2**。因此它的新特徵是
$x_2 = \exp(-0.3 \times 1^2) \approx 0.74$ 且 $x_3 = \exp(-0.3 \times 2^2) \approx 0.30$ ——
一維的點 $x_1$ 就這樣變成了二維的 $(x_2, x_3)$:

![用兩個地標的相似度把一維資料映射到二維](./img/6.png)

程式重現上面這張圖:

In [ ]:
def gaussian_rbf(x, landmark, gamma):
    '''RBF 相似度:exp(-gamma * 距離^2)'''
    return np.exp(-gamma * np.linalg.norm(x - landmark, axis=1)**2)

gamma = 0.3

x1s = np.linspace(-4.5, 4.5, 200).reshape(-1, 1)
x2s = gaussian_rbf(x1s, -2, gamma)   # 到地標 1(x=-2)的相似度曲線
x3s = gaussian_rbf(x1s, 1, gamma)    # 到地標 2(x=1)的相似度曲線

XK = np.c_[gaussian_rbf(X1D, -2, gamma), gaussian_rbf(X1D, 1, gamma)]   # 9 個點的新特徵 (x2, x3)
yk = np.array([0, 0, 1, 1, 1, 1, 1, 0, 0])

plt.figure(figsize=(11, 4))

plt.subplot(121)                     # 左圖:原始一維資料 + 兩條相似度曲線
plt.grid(True, which='both')
plt.axhline(y=0, color='k')
plt.scatter(x=[-2, 1], y=[0, 0], s=150, alpha=0.5, c="red")   # 紅點:兩個地標
plt.plot(X1D[:, 0][yk == 0], np.zeros(4), "bs")
plt.plot(X1D[:, 0][yk == 1], np.zeros(5), "g^")
plt.plot(x1s, x2s, "g--")            # 綠虛線:到地標 1 的相似度
plt.plot(x1s, x3s, "b:")             # 藍點線:到地標 2 的相似度
plt.gca().get_yaxis().set_ticks([0, 0.25, 0.5, 0.75, 1])
plt.xlabel(r"$x_1$", fontsize=20)
plt.ylabel(r"Similarity", fontsize=14)
plt.annotate(r'$\mathbf{x}$',
             xy=(X1D[3, 0], 0),
             xytext=(-0.5, 0.20),
             ha="center",
             arrowprops=dict(facecolor='black', shrink=0.1),
             fontsize=18,
            )
plt.text(-2, 0.9, "$x_2$", ha="center", fontsize=20)
plt.text(1, 0.9, "$x_3$", ha="center", fontsize=20)
plt.axis([-4.5, 4.5, -0.1, 1.1])

plt.subplot(122)                     # 右圖:映射到 (x2, x3) 平面
plt.grid(True, which='both')
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.plot(XK[:, 0][yk == 0], XK[:, 1][yk == 0], "bs")
plt.plot(XK[:, 0][yk == 1], XK[:, 1][yk == 1], "g^")
plt.xlabel(r"$x_2$", fontsize=20)
plt.ylabel(r"$x_3$  ", fontsize=20, rotation=0)
plt.annotate(r'$\phi\left(\mathbf{x}\right)$',
             xy=(XK[3, 0], XK[3, 1]),
             xytext=(0.65, 0.50),
             ha="center",
             arrowprops=dict(facecolor='black', shrink=0.1),
             fontsize=18,
            )
plt.plot([-0.1, 1.1], [0.57, -0.1], "r--", linewidth=3)   # 映射後,一條直線就分開了
plt.axis([-0.1, 1.1, -0.1, 1.1])

plt.subplots_adjust(right=1)
plt.show()

**觀察結果:** 左圖的一維資料原本不可分;右圖以「跟兩個地標的相似度」$(x_2, x_3)$ 為新座標後,
綠三角和藍方塊被紅色虛線(一條直線)乾淨分開 —— 相似度特徵完成了升維。

**那地標要放在哪?理論情況下會得到多少維特徵呢?** 最直接的做法:**對每一個實例(樣本資料點)都建立一個地標** ——
此時會將 $m \times n$ 的訓練集轉換成 $m \times m$ 的訓練集(每筆樣本的新特徵 = 跟所有 m 筆樣本的相似度)。

> 💡 樣本一多,$m \times m$ 根本存不下、算不動 —— 幸好 **SVM 利用了核函數的計算技巧,
> 大大降低了計算複雜度**:`SVC(kernel="rbf")` 等同於在這個超高維空間裡訓練,卻從不真正生成這些特徵。

### 5.3 超參數 $\gamma$:控制高斯曲線的寬窄

- **增加 $\gamma$** 使高斯曲線變**窄**,每個實例的影響範圍都較小:決策邊界最終變得更不規則,在個別實例周圍擺動
- **減少 $\gamma$** 使高斯曲線變**寬**,實例具有更大的影響範圍,決策邊界更加平滑

先訓練一個 RBF 核的 SVM 試試:

In [ ]:
rbf_kernel_svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm_clf", SVC(kernel="rbf", gamma=5, C=0.001))   # 高斯核:gamma 窄、C 很寬鬆
])

rbf_kernel_svm_clf.fit(X, y)

接著做一次網格對比實驗:$\gamma \in \{0.1, 5\}$ × $C \in \{0.001, 1000\}$,四種組合各訓練一個模型:

In [ ]:
gamma1, gamma2 = 0.1, 5
C1, C2 = 0.001, 1000
hyperparams = (gamma1, C1), (gamma1, C2), (gamma2, C1), (gamma2, C2)

svm_clfs = []
for gamma, C in hyperparams:                  # 四種超參數組合,各訓練一個 RBF-SVM
    rbf_kernel_svm_clf = Pipeline([
            ("scaler", StandardScaler()),
            ("svm_clf", SVC(kernel="rbf", gamma=gamma, C=C))
        ])
    rbf_kernel_svm_clf.fit(X, y)
    svm_clfs.append(rbf_kernel_svm_clf)

plt.figure(figsize=(11, 7))

for i, svm_clf in enumerate(svm_clfs):
    plt.subplot(221 + i)
    plot_predictions(svm_clf, [-1.5, 2.5, -1, 1.5])
    plot_dataset(X, y, [-1.5, 2.5, -1, 1.5])
    gamma, C = hyperparams[i]
    plt.title(r"$\gamma = {}, C = {}$".format(gamma, C), fontsize=16)
plt.show()

**觀察結果(四宮格):**

- **上排($\gamma=0.1$,曲線寬)**:邊界平滑;搭配大 C(右上)後貼合得更積極一些
- **下排($\gamma=5$,曲線窄)**:邊界繞著個別樣本擺動,形成不規則的「島」—— 明顯過擬合的長相
- C 的作用和第 3 部分一致:越大越不容忍錯誤

> 💡 $\gamma$ 的行為就像一個正規化超參數:**模型過擬合就減小 $\gamma$,欠擬合就增大 $\gamma$**(常和 C 一起調)。

## 總結:這份實驗的重點整理

| # | 重點 |
|:---:|---|
| 1 | SVM 是**最大間隔分類器**:在分對的前提下,選離兩類都最遠的邊界;邊界只由**支持向量**決定 |
| 2 | SVM 以距離為核心,**對特徵尺度敏感 → 訓練前必先標準化** |
| 3 | **軟間隔**允許少量樣本越線來換取更寬的間隔;**C 越大越嚴格**(間隔小、易過擬合),過擬合時調小 C |
| 4 | 線性不可分 → **升維**(如加 $x^2$、多項式特徵)後就可能線性可分 |
| 5 | **核技巧**:不真正生成高維特徵,直接算出升維後的內積 —— 效果等同升維,計算量大減 |
| 6 | **高斯核(RBF)** 用「與地標的相似度」造特徵;**$\gamma$ 越大曲線越窄、邊界越不規則**,過擬合時調小 $\gamma$ |

**多做實驗,得出結果!!!**